# 22 — PromptOps

## Scenario
We are deploying a sentiment analysis prompt to production. 

**The Problem:** A developer manually edits the prompt string in the production codebase to "make it better," but accidentally breaks the JSON schema output. The app goes down.

**The Solution:** We implement **PromptOps**. A prompt is no longer just a string; it is a versioned **Behavior Artifact** that includes the prompt, the schema, and a test suite. It must pass a CI/CD Release Gate before deployment.

In [ ]:
import os
from typing import List, Dict, Callable
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: Defining the Behavior Artifact

A Behavior Artifact packages everything required for the prompt to run safely in production.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: str = Field(description="Must be exactly 'POSITIVE', 'NEGATIVE', or 'NEUTRAL'")

class BehaviorArtifact:
    def __init__(self, version: str, prompt_text: str, schema: type[BaseModel], test_suite: List[Dict]):
        self.version = version
        self.prompt_text = prompt_text
        self.schema = schema
        self.test_suite = test_suite

# Our standardized test suite
sentiment_tests = [
    {"input": "I love this product!", "expected": "POSITIVE"},
    {"input": "This is the worst thing ever.", "expected": "NEGATIVE"},
    {"input": "It arrived on Tuesday.", "expected": "NEUTRAL"}
]


## Step 2: The CI/CD Release Gate

Before a Behavior Artifact can be "deployed", it must pass this evaluation script.

In [ ]:
def release_gate(artifact: BehaviorArtifact, required_accuracy: float = 1.0) -> bool:
    print(f"\n--- Running CI/CD Gate for {artifact.version} ---")
    passed_tests = 0
    total_tests = len(artifact.test_suite)
    
    for test in artifact.test_suite:
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=f"{artifact.prompt_text}\nText: {test['input']}",
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=artifact.schema,
                )
            )
            result = artifact.schema.model_validate_json(response.text)
            
            if result.sentiment == test['expected']:
                passed_tests += 1
            else:
                print(f"Test Failed: Expected {test['expected']}, got {result.sentiment}")
        except Exception as e:
            print(f"Test Crashed (Schema Violation?): {e}")
            
    accuracy = passed_tests / total_tests
    print(f"Accuracy: {accuracy * 100:.1f}%")
    
    if accuracy >= required_accuracy:
        print("✅ RELEASE GATE PASSED. Deploying to production.")
        return True
    else:
        print("❌ RELEASE GATE FAILED. Deployment blocked.")
        return False


## Step 3: Testing the System

A developer submits a "clever" new prompt that breaks the rules (v1.1). The Release Gate blocks it.
They fix it and submit v1.2, which passes.

In [ ]:
# A bad update: The developer tells the model to output a number instead of the required strings!
bad_update = BehaviorArtifact(
    version="v1.1-bad-update",
    prompt_text="Analyze sentiment. Return 1 for positive, -1 for negative, 0 for neutral.",
    schema=SentimentResult, # The schema expects 'POSITIVE', 'NEGATIVE', 'NEUTRAL'
    test_suite=sentiment_tests
)
release_gate(bad_update)

# A good update: The developer improves the prompt safely.
good_update = BehaviorArtifact(
    version="v1.2-good-update",
    prompt_text="You are a strict sentiment analyzer. Return POSITIVE, NEGATIVE, or NEUTRAL only.",
    schema=SentimentResult,
    test_suite=sentiment_tests
)
release_gate(good_update)


## Conclusion

Prompt engineering in the enterprise is indistinguishable from software engineering.

You must use **PromptOps** to package your prompts, schemas, and tests together, and run them through automated CI/CD Release Gates before they ever reach production.